In [28]:
import pandas as pd
import glob

# =====================================================================
# 1. UNIFICAR LOS 3 ARCHIVOS DEL RANKING FIFA
# =====================================================================

# Asumiendo que tus archivos se llaman algo como ranking_1.csv, ranking_2.csv...
# Si sabes sus nombres exactos, puedes cargarlos uno a uno y usar pd.concat
archivos_ranking = [
    'data/fifa_ranking-2023-07-20.csv',  # Cambia estos nombres por tus rutas reales
    'data/fifa_ranking-2024-04-04.csv',
    'data/fifa_ranking-2024-06-20.csv'
]

# Leemos y concatenamos los tres archivos en uno solo
df_ranking_completo = pd.concat([pd.read_csv(f) for f in archivos_ranking], ignore_index=True)

# =====================================================================
# 2. LIMPIEZA Y PREPARACIÓN CRUCIAL PARA EL MERGE_ASOF
# =====================================================================

# Convertimos la fecha a formato datetime
df_ranking_completo['rank_date'] = pd.to_datetime(df_ranking_completo['rank_date'])

# Nos quedamos solo con las columnas que aportan valor al modelo
df_ranking_limpio = df_ranking_completo[['rank_date', 'country_full', 'total_points', 'rank']].copy()

# REGLA DE ORO: Para merge_asof, el DataFrame TIENE que estar ordenado por fecha
df_ranking_limpio = df_ranking_limpio.sort_values('rank_date')

print(f"✅ Rankings unificados. Total de filas históricas: {df_ranking_limpio.shape[0]}")
df_ranking_limpio

✅ Rankings unificados. Total de filas históricas: 199490


,rank_date,country_full,total_points,rank
0,1992-12-31,Germany,57.00,1.0
64860,1992-12-31,Korea DPR,16.00,77.0
64861,1992-12-31,Peru,16.00,78.0
64862,1992-12-31,Sierra Leone,16.00,79.0
64863,1992-12-31,Tanzania,15.00,80.0
...,...,...,...,...
199352,2024-06-20,France,1837.47,2.0
199353,2024-06-20,Madagascar,1203.66,104.0
199354,2024-06-20,Morocco,1669.44,12.0
199356,2024-06-20,Poland,1541.49,26.0


In [29]:
df_partidos = pd.read_csv('data/results.csv', parse_dates=['date'])
df_partidos = df_partidos.sort_values('date').reset_index(drop=True)
df_partidos


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49282,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49283,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49284,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,Kansas City,United States,True
49285,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True


In [30]:
# Filtramos los partidos para empezar a partir del 1 de enero de 1993
df_partidos = df_partidos[df_partidos['date'] >= '1993-01-01'].copy()

# Ordenamos ambos DataFrames cronológicamente, esto es obligatorio para merge_asof
df_partidos = df_partidos.sort_values('date')
df_ranking_limpio = df_ranking_limpio.sort_values('rank_date')

# =====================================================================
# 3. CRUCE INTELIGENTE (MERGE_ASOF) POR BLOQUES
# =====================================================================

# Cruce para el Equipo Local
df_unificado = pd.merge_asof(
    df_partidos, 
    df_ranking_limpio[['rank_date', 'country_full', 'total_points', 'rank']], 
    left_on='date', 
    right_on='rank_date', 
    left_by='home_team', 
    right_by='country_full', 
    direction='backward'  # Busca el último ranking publicado antes del partido
)

df_unificado = df_unificado.rename(columns={'total_points': 'puntos_fifa_local', 'rank': 'ranking_pos_local'})
df_unificado = df_unificado.drop(columns=['rank_date', 'country_full'], errors='ignore')

# Cruce para el Equipo Visitante
df_unificado = pd.merge_asof(
    df_unificado, 
    df_ranking_limpio[['rank_date', 'country_full', 'total_points', 'rank']], 
    left_on='date', 
    right_on='rank_date', 
    left_by='away_team', 
    right_by='country_full', 
    direction='backward'
)
df_unificado = df_unificado.rename(columns={'total_points': 'puntos_fifa_visitante', 'rank': 'ranking_pos_visitante'})
df_unificado = df_unificado.drop(columns=['rank_date', 'country_full'], errors='ignore')

# =====================================================================
# 4. PROTECCIÓN PARA LOS PARTIDOS RECIENTES Y DEL MUNDIAL (2025 - 2026)
# =====================================================================
# Como el ranking llega a 2024, los partidos de 2025/2026 tendrán NaN. 
# Con '.ffill()' le decimos a Pandas: "Rellena los NaN de cada equipo con su último ranking conocido flotante hacia adelante"
df_unificado['puntos_fifa_local'] = df_unificado.groupby('home_team')['puntos_fifa_local'].ffill()
df_unificado['puntos_fifa_visitante'] = df_unificado.groupby('away_team')['puntos_fifa_visitante'].ffill()

df_unificado['ranking_pos_local'] = df_unificado.groupby('home_team')['ranking_pos_local'].ffill()
df_unificado['ranking_pos_visitante'] = df_unificado.groupby('away_team')['ranking_pos_visitante'].ffill()

# Si queda alguna selección muy pequeña que jamás haya entrado en los rankings, le ponemos una base neutral
df_unificado['puntos_fifa_local'] = df_unificado['puntos_fifa_local'].fillna(df_unificado['puntos_fifa_local'].mean())
df_unificado['puntos_fifa_visitante'] = df_unificado['puntos_fifa_visitante'].fillna(df_unificado['puntos_fifa_visitante'].mean())

# =====================================================================
# 5. INGENIERÍA DE VARIABLES DEL RANKING
# =====================================================================
df_unificado['dif_puntos_fifa'] = df_unificado['puntos_fifa_local'] - df_unificado['puntos_fifa_visitante']
df_unificado

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,puntos_fifa_local,ranking_pos_local,puntos_fifa_visitante,ranking_pos_visitante,dif_puntos_fifa
0,1993-01-01,Ghana,Mali,1.0,1.0,Friendly,Libreville,Gabon,True,34.000000,39.0,22.00,69.0,12.000000
1,1993-01-02,Gabon,Burkina Faso,1.0,1.0,Friendly,Libreville,Gabon,False,27.000000,55.0,11.00,97.0,16.000000
2,1993-01-02,Kuwait,Lebanon,2.0,0.0,Friendly,Kuwait City,Kuwait,False,21.000000,71.0,0.00,161.0,21.000000
3,1993-01-03,Burkina Faso,Mali,1.0,0.0,Friendly,Libreville,Gabon,True,11.000000,97.0,22.00,69.0,-11.000000
4,1993-01-03,Gabon,Ghana,2.0,3.0,Friendly,Libreville,Gabon,False,27.000000,55.0,34.00,39.0,-7.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30578,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,1482.100000,43.0,1787.88,5.0,-305.780000
30579,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,1374.130000,68.0,1860.14,1.0,-486.010000
30580,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,631.567156,NaN,1397.41,62.0,-765.842844
30581,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,1669.440000,12.0,1747.04,6.0,-77.600000


In [31]:
import pandas as pd
import numpy as np

# Asumimos que tu DataFrame unificado con los rankings se llama df_unificado
df_variables = df_unificado.copy()

# =====================================================================
# 1. DEFINIR LA VARIABLE OBJETIVO (RESULTADO DEL ENCUENTRO)
# =====================================================================
condiciones = [
    (df_variables["home_score"] > df_variables["away_score"]),  # Gana Local -> 1
    (df_variables["home_score"] == df_variables["away_score"]),  # Empate -> 0
    (df_variables["home_score"] < df_variables["away_score"]),  # Gana Visitante -> 2
]
opciones = [1, 0, 2]

# Asignamos el resultado (Los partidos de 2026 quedarán con NaN de forma natural)
df_variables["resultado"] = np.select(condiciones, opciones, default=np.nan)

# =====================================================================
# 2. CREAR LA LÍNEA DE TIEMPO ÚNICA POR EQUIPO PARA RACHAS
# =====================================================================
locales = df_variables[["date", "home_team", "home_score", "away_score"]].rename(
    columns={
        "home_team": "equipo",
        "home_score": "goles_marcados",
        "away_score": "goles_concedidos",
    }
)
locales["puntos"] = np.select(
    [
        locales["goles_marcados"] > locales["goles_concedidos"],
        locales["goles_marcados"] == locales["goles_concedidos"],
    ],
    [3, 1],
    default=0,
)

visitantes = df_variables[["date", "away_team", "away_score", "home_score"]].rename(
    columns={
        "away_team": "equipo",
        "away_score": "goles_marcados",
        "home_score": "goles_concedidos",
    }
)
visitantes["puntos"] = np.select(
    [
        visitantes["goles_marcados"] > visitantes["goles_concedidos"],
        visitantes["goles_marcados"] == visitantes["goles_concedidos"],
    ],
    [3, 1],
    default=0,
)

# Unimos todo cronológicamente por equipo
linea_tiempo = pd.concat([locales, visitantes]).sort_values(["equipo", "date"])

# =====================================================================
# 3. CALCULAR MÉTRICAS MÓVILES (GOLES TOTALES EN LOS ÚLTIMOS 10 PARTIDOS)
# =====================================================================
group = linea_tiempo.groupby("equipo")

# CAMBIO CLAVE: Usamos .sum() en lugar de .mean() para obtener goles totales
linea_tiempo["goles_marcados_10"] = group["goles_marcados"].transform(
    lambda x: x.shift(1).rolling(10, min_periods=1).sum()
)
linea_tiempo["goles_concedidos_10"] = group["goles_concedidos"].transform(
    lambda x: x.shift(1).rolling(10, min_periods=1).sum()
)
linea_tiempo["forma_puntos_10"] = group["puntos"].transform(
    lambda x: x.shift(1).rolling(10, min_periods=1).sum()
)

# Limpiamos duplicados en la línea de tiempo por seguridad
linea_tiempo_limpia = linea_tiempo[
    ["date", "equipo", "goles_marcados_10", "goles_concedidos_10", "forma_puntos_10"]
].drop_duplicates(subset=["date", "equipo"])

# =====================================================================
# 4. FUSIONAR DE VUELTA AL DATAFRAME PRINCIPAL
# =====================================================================
# Cruzamos datos para el Local
df_variables = df_variables.merge(
    linea_tiempo_limpia,
    left_on=["date", "home_team"],
    right_on=["date", "equipo"],
    how="left",
)
df_variables = df_variables.rename(
    columns={
        "goles_marcados_10": "goles_totales_marcados_local_10",
        "goles_concedidos_10": "goles_totales_concedidos_local_10",
        "forma_puntos_10": "forma_puntos_local_10",
    }
).drop(columns=["equipo"])

# Cruzamos datos para el Visitante
df_variables = df_variables.merge(
    linea_tiempo_limpia,
    left_on=["date", "away_team"],
    right_on=["date", "equipo"],
    how="left",
)
df_variables = df_variables.rename(
    columns={
        "goles_marcados_10": "goles_totales_marcados_visitante_10",
        "goles_concedidos_10": "goles_totales_concedidos_visitante_10",
        "forma_puntos_10": "forma_puntos_visitante_10",
    }
).drop(columns=["equipo"])

# =====================================================================
# 5. INGENIERÍA DE VARIABLES PREDICTIVAS DE DIFERENCIAS EN ABSOLUTO
# =====================================================================
df_variables["dif_goles_marcados_10"] = (
    df_variables["goles_totales_marcados_local_10"]
    - df_variables["goles_totales_marcados_visitante_10"]
)
df_variables["dif_goles_concedidos_10"] = (
    df_variables["goles_totales_concedidos_local_10"]
    - df_variables["goles_totales_concedidos_visitante_10"]
)
df_variables["dif_forma_puntos_10"] = (
    df_variables["forma_puntos_local_10"] - df_variables["forma_puntos_visitante_10"]
)


# Distinguir partidos oficiales de amistosos
def mapear_importancia_torneo(torneo):
    torneo = str(torneo)
    if "FIFA World Cup" in torneo and "FIFA World Cup qualification" not in torneo:
        return 4  # El Mundial absoluto
    elif (
        "Copa América" in torneo
        or "UEFA Euro" in torneo
        or "African Cup of Nations" in torneo
        and "qualification" not in torneo
    ):
        return 3  # Torneos continentales mayores
    elif (
        "FIFA World Cup qualification" in torneo
        or "nations league" in torneo
        or "Confederations Cup" in torneo
    ):
        return 2  # Clasificatorios y torneos oficiales menores
    elif "Friendly" in torneo:
        return 1  # Amistosos
    else:
        return 1  # Otros torneos menores (Copas de invitación, etc.)


df_variables["importancia_torneo"] = df_variables["tournament"].apply(
    mapear_importancia_torneo
)

# Rellenar nulos iniciales con 0
columnas_rachas = [
    "goles_totales_marcados_local_10",
    "goles_totales_concedidos_local_10",
    "forma_puntos_local_10",
    "goles_totales_marcados_visitante_10",
    "goles_totales_concedidos_visitante_10",
    "forma_puntos_visitante_10",
    "dif_goles_marcados_10",
    "dif_goles_concedidos_10",
    "dif_forma_puntos_10",
]
for col in columnas_rachas:
    df_variables[col] = df_variables[col].fillna(0)

print("🎯 ¡Métricas de goles totales calculadas con éxito!")
df_variables
df_variables.to_csv("data/partidos_entrenamiento.csv", index=False)

🎯 ¡Métricas de goles totales calculadas con éxito!


In [32]:
print(df_variables['resultado'].value_counts(normalize=True) * 100)

resultado
1.0    48.513651
2.0    28.147226
0.0    23.339124
Name: proportion, dtype: float64


In [33]:
print(pd.crosstab(df_variables['neutral'], df_variables['resultado'], normalize='index') * 100)

resultado        0.0        1.0        2.0
neutral                                   
False      23.275035  50.851637  25.873328
True       23.502090  42.568509  33.929401


In [34]:
print(df_variables['neutral'].value_counts())

neutral
False    21908
True      8675
Name: count, dtype: int64


In [35]:
display(df_variables[['tournament', 'importancia_torneo']].head(20))

,tournament,importancia_torneo
0,Friendly,1
1,Friendly,1
2,Friendly,1
3,Friendly,1
4,Friendly,1
5,Friendly,1
6,Friendly,1
7,Friendly,1
8,African Cup of Nations qualification,1
9,FIFA World Cup qualification,2
